# TN2211 Session 7

## Instruments


In [ ]:
import sys
sys.path.append("../drivers/")
from tn2211_drivers import *
import glob
import matplotlib.pyplot as plt
import math
import numpy as np

In [ ]:
import pyvisa
rm = pyvisa.ResourceManager()
rm.list_resources()

In [ ]:
scope = Scope("SDS")
gen = Generator("SDG")

# Diode IV curves


Configure scope and generator with a reasonable set of defaults:

In [ ]:
# Configure the channels we will use
scope.write("CHAN1:SWIT ON")
scope.write("CHAN2:SWIT ON")
scope.write("CHAN3:SWIT ON") # Use the SYNC of the generator for chan3, and trigger of that
scope.write("CHAN1:COUP DC")
scope.write("CHAN2:COUP DC")

# This will allow voltages at the top of the diode to go up to 10V
# Since there is a series resistor in the source, and also a series
# resistor we are using th read out the current, the voltage across
# the diode will be much less.
scope.write("CHAN1:SCAL 1")
scope.write("CHAN2:SCAL 1")

 # it is best to keep offset to Zero (Gary saw small distortions of the diode IVs if not zero...)
scope.write("CHAN1:OFF 0") 
scope.write("CHAN2:OFF 0") 
scope.write("FUNC1 OFF")

# We will pick a ramp with a relatively slow frequency so that it is really
# measuring the "static" IV characteristics of the diode, and not sensitive 
# to things like the diode capacitance. We will also pick a timebase that 
# will display a few ramps on the screen. If you set this higher, you will
# start to see strange thigns in your IV due to the capacitances causing low
# pass filters which result in the current lagging in time compared to what 
# the ramp is setting with the generator. 1 kHz ramp repeat rate should be
# safe. 
gen.write("C1:BSWV WVTP,RAMP,FRQ,1e3")
scope.write("TIM:SCAL .2e-3")

A function to acquire the traces and convert the two measured voltages into the voltage across the diode and the current through the diode based on a 1 kohm readout resistor. 

For the DC curve, we use a 100 ohm "current readout" resistor to be able to get more current through the diode.

In [ ]:
def get_diode_curve(v1, v2):
    '''Assumes that you have connected Ch1 of scope above the diode and Ch2 of scope
    above the 10 ohm current readout resistor. You should adjust the scope vertical
    and horizontal settings to be able to see both voltage time traces clearly.

    This will generate a diode IV by sweeping the generator voltage from v1 to v2. 
    '''
    print("=== Make sure both Ch1 and Ch2 are DC coupled!!! ===\n")
    # Set generator range
    gen.set_offset(1,(v1+v2)/2)
    gen.set_amplitude(1,v2-v1)
    t,v1 = scope.get_trace(1, npoints="all")
    t,v2 = scope.get_trace(2, npoints="all")
    v = v1 - v2
    R = 100 # the value of the readout resistor
    i = v2/R 
    
    # Handy to have them sorted by voltage (x-axis)
    ind = np.argsort(v)
    v = v[ind]
    i = i[ind]
    
    return v,i

In [ ]:
### Diode comparison

Perform a sweep, download the Ch1 and Ch2 curves, and convert them into diode current and diode voltage:

In [ ]:
# Change the voltage range to explore what your diode looks like
# If the voltage trace you see on the scope goes off scale, you will
# need to manually adjust the scope ranges 
v,i = get_diode_curve(-2,4) 
plt.plot(v,i, '.', ms=0.3)
plt.ylabel("Current through diode (A)")
plt.xlabel("Votlage across diode (V)")
plt.title("Gary's 1N4148 diode curve") # Change title if try different diodes and take out Gary's name

Repeating this for different diodes in your box, we can plot them all on top of each other to compare them: 

In [ ]:
diode1_type = "type"
v1,i1 = get_diode_curve(-10,10) 

In [ ]:
diode2_type = "type"
v2, i2 = get_diode_curve(-10,10)

In [ ]:
diode3_type = "type"
v3, i3 = get_diode_curve(-10,10)

In [ ]:
diode4_type = "type"
v4, i4 = get_diode_curve(-10,10)

In [ ]:
diode5_type = "type"
v5, i5 = get_diode_curve(-10,10)

In [ ]:
diode6_type = "type"
v6, i6 = get_diode_curve(-10,10)

In [ ]:
plt.plot(v1,i1, '.', ms=0.3, label=diode1)
plt.plot(v2,i2, '.', ms=0.3, label=diode1)
plt.plot(v3,i3, '.', ms=0.3, label=diode1)
plt.plot(v4,i4, '.', ms=0.3, label=diode1)
plt.plot(v5,i5, '.', ms=0.3, label=diode1)
plt.plot(v6,i6, '.', ms=0.3, label=diode1)

plt.ylabel("Current through diode (A)")
plt.xlabel("Votlage across diode (V)")
plt.title("Comparison of different diode types") # Change title if try different diodes and take out Gary's name

### Comparison of 1N4148 to Shockley Model

Shockley Model:

$$
I = I_s \left ( e^{V/nV_T} - 1 \right) 
$$

In [ ]:
def shockely_model(v, Is, n):
    return Is * (np.exp(v/n/25e-3)-1)

For plotting against the model, this is a useful ranget to sweep:

In [ ]:
v,i = get_diode_curve(-0.5,4) 

If you look carefully, there will be a small offset from zero that is from offset voltages in the amplifier used by the scope:

In [ ]:
plt.plot(v,i, '.', ms=0.3)
plt.ylabel("Current through diode (A)")
plt.xlabel("Votlage across diode (V)")
plt.title("Gary's 1N4148 diode curve") # Change title if try different diodes and take out Gary's name
plt.axhline(0,ls=':', c='grey')
plt.ylim(-0.5e-3,0.5e-3)

To compare to the Shockely model, we will first subtract this offset. The saturation current is likely on the order of nA, which will be well below our noise floor, we we can just manually pick a range where the curve looks flat to estimate our current offset, and then compare to the model after this subtraction:

In [ ]:
# Conditions to use for estimating the offset current in our measurement;
ind = np.where(v<0.3)
i_off = np.average(i[ind])

# You can also uncomment this to see what it looks like if you do not correct the offset
# i_off = 0

# What worked well for Gary's 1n4148 diode
Is = 4e-9 # saturation current in amps (Gary found 4.0 nA worked well)
n = 2.1 # ideality factor (Gary found 2.1 worked well)

plt.title("Gary's 1N4148 Diode curve") # replace when you test the other diodes, and take out Gary's name :)
plt.plot(v,i-i_off, '.', ms=0.5, label="Data")
plt.yscale('log')
plt.plot(v, shockely_model(v,Is,n), label = "Shockely model Is = %.2e A, n = %.2f" % (Is, n))
plt.ylim(1e-6,) # adjust to prevent model from squeezing vertical scale to much
plt.ylabel("Current through diode (A)")
plt.xlabel("Votlage across diode (V)")
plt.legend()
plt.show()

# Diode rectifier

In [ ]:
# See instructions for what to set up here. You need to only get screenshots for your report. 
scope.get_screenshot()

# Diode small signal response

Configure the instrument settings:

In [ ]:
scope.write("CHAN1:COUP AC")
scope.write("CHAN2:COUP AC")
scope.write("CHAN3:COUP DC")
scope.write("CHAN1:SCAL 5e-3")
scope.write("CHAN2:SCAL 5e-3")
scope.write("CHAN3:SCAL 0.5")
scope.write("TIM:SCAL 20e-6")
gen.write("C1:BSWV WVTP,SINE,FRQ,100e3,AMP,20e-3")
gen.write("C1:OUTP ON")

In [ ]:
import time

v_diode = []
i = []
I_dc= []

# The readout resistor. A higher readout resistor will give you a 
# smaller absolute current you can measure (more voltage for the same current)
# but will limit how far you can current bias the diode
R = 100 
# R = 1000 

# Number of points. The speed of the data acquisition is limited by the 
# 200 ms rise time of the AC coupling filter of the scope. 21 points takes about
# 20 seconds, which is good for first tests. For your best data, you can set this 
# to more points for a "high res" trace for your analysis logbook, limited 
# only by your paitence...
Npoints = 21

# It is better not to go above 2.9 V since a 3.0 V the generator 
# changes its internal gain and gets noisier
v_offset = np.linspace(-0.5,2.9,21)

# Let it settle before we start the sweep
gen.set_offset(1,v_offset[0])
time.sleep(5)

for v_off in v_offset:
    print(".", end="") # A simple status indicator :)
    gen.set_offset(1,v_off)
    time.sleep(0.7) # wait 0.7 seconds for ac coupling transient to settle
    # Measure the "small signal" voltages from the amplitude of the oscillations
    v1 = np.std(scope.get_trace(1, save_file=False)) # use the standard deviation function to calculate the RMS voltage oscillation amplitude
    v2 = np.std(scope.get_trace(2, save_file=False))
    v2_dc = np.average(scope.get_trace(3, save_file=False))
    v_diode.append(v1-v2) # v1 at top of diode, v2 at top of resistor
    I_dc.append(v2_dc/R)
    i.append(v2 / R)   

In [ ]:
v_diode = np.array(v_diode)
i = np.array(i)
r = v_diode / i

title= "Small signal response of 1N4148 diode (Gary)" # Remove my name, change title as appropriate

plt.figure(figsize=(12,4))
plt.subplot(121)
plt.title(title)
plt.plot(v_offset, r)
plt.yscale("log")
plt.axhline(50,ls=":",c='gray')
plt.ylabel("Small signal resistance r (ohms)")
plt.xlabel("DC Voltage source offset (V)")
plt.subplot(122)
plt.plot(I_dc, r)
plt.yscale("log")
plt.axhline(50,ls=":",c='gray')
plt.ylabel("Small signal resistance r (ohms)")
plt.xlabel("DC Diode current (A)")
plt.title(title)

# A diode mixer

Set up the scope and all the sources:

In [ ]:
scope.write("CHAN1:SWIT ON")
scope.write("CHAN2:SWIT OFF")
scope.write("CHAN3:SWIT OFF")

scope.write("CHAN1:SCAL 20e-3")
scope.write("CHAN2:SCAL 20e-3")
scope.write("TIM:SCAL 200e-6")

def set_freq_range(f1, f2):
    center = (f1+f2)/2
    span = f2-f1
    scope.write("FUNC1:FFT:SPAN %e" % span)
    scope.write("FUNC1:FFT:HCEN %e" % center)

scope.write("FUNC:FFTD EXCL")

scope.write("FUNC1 ON")
scope.write("FUNC1:SOUR1 C1")
scope.write("FUNC1:OPER FFT")
scope.write("FUNC:FFT:MODE AVER,4")   # Better signal to noise on our spectrum
set_freq_range(0,2.3e6)

# Also configure some convenient FFT Marker setttings
# If you explore a bit, you can reconfigure these, either by changing 
# the code here or by using the buttons on the scope
scope.write("FUNC1:FFT:SEAR PEAK")
scope.write("FUNC1:FFT:SEAR:EXC 10")
scope.write("FUNC1:FFT:SEAR:THR -70")

In [ ]:
gen.write("C1:BSWV WVTP,SINE,FRQ,1e6,AMP,100e-3")
gen.write("C2:BSWV WVTP,SINE,FRQ,1.1e6,AMP,100e-3")

In [ ]:
def get_peaks(print_results=True):
    res = scope.query("FUNC1:FFT:SEAR:RES?")
    res = res.strip("Peaks,").rstrip().split(";")[:-1]
    if print_results:
        print("Peaks:\n")
    f = []
    dBv = []
    for r in res:
        r = r.split(",")
        f.append(float(r[1]))
        dBv.append(float(r[2]))
        if print_results:
            print("%s  % 10.3f kHz % 10.3f dBm" % (r[0], float(r[1])/1e3, float(r[2])))
    return np.array(f), np.array(dBv)

This cell will get the currently displayed FFT and found peaks, and make a plot that you can use in your logbook:

In [ ]:
voffset = 0.0
gen.set_offset(2,voffset)
time.sleep(5) # for averaging 

In [ ]:
# You may want to adjust this if your peaks are small, depending on how much you current 
# bias your diode
scope.write("FUNC1:FFT:SEAR:THR -80") 

In [ ]:
freq, dBv, _ = scope.get_fft(1)
get_peaks()
plt.figure(figsize=(12,4))
plt.plot(freq, dBv)
plt.xlabel("Freq (Hz)")
plt.ylabel("Spectrum (dBV)")
plt.title("Put here a useful title for your logbook, use variables eg Voff = %.3f" % (voffset))
plt.show()

## Level 2 work

Add your own code and code cells below for your level 2 work. 

In [ ]:
# Your level 2 code. Take screenshots for your logbook.